# 02 — Data Cleaning
This notebook performs all column-level cleaning on the raw `airbnb_listing.csv` dataset.

Our analysis has **3 perspectives**:
1. **Business** — Where should someone open a new hotel? What pricing, property type, and amenities drive revenue?
2. **User** — What do guests care about? (ratings, reviews, location)
3. **Airbnb Growth** — How is the platform expanding over time?

We drop columns in two passes:
- **Pass 1**: Scraping metadata, internal IDs/URLs, redundant geography, overly-granular booking rules, duplicate host counts, legal fields.
- **Pass 2**: Personal host profile text, free-text descriptions, redundant date/availability fields — columns that can't be used in quantitative business analysis.

In [ ]:
import pandas as pd

# Load the raw dataset
file_path = '../data/raw/airbnb_listing.csv'
df = pd.read_csv(file_path, low_memory=False)
print(f'Loaded dataset: {df.shape[0]} rows × {df.shape[1]} columns')

## Pass 1 — Drop Scraping Metadata, IDs, URLs & Redundant Columns
These columns are system artifacts from the web-scraping process, internal identifiers, image URLs, or mathematically redundant duplicates of other columns.

In [ ]:
pass1_cols = [
    # Scraping / system metadata
    'scrape_id', 'last_scraped', 'source', 'calendar_updated', 'calendar_last_scraped',
    # Internal IDs & URLs
    'id', 'listing_url', 'host_id', 'host_url',
    # Image URLs
    'picture_url', 'host_thumbnail_url', 'host_picture_url',
    # Overly-granular booking rules (we keep minimum_nights & maximum_nights)
    'minimum_minimum_nights', 'maximum_minimum_nights',
    'minimum_maximum_nights', 'maximum_maximum_nights',
    'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm',
    # Redundant / highly-missing geography
    'neighbourhood_group_cleansed', 'host_neighbourhood', 'neighbourhood',
    # Redundant calculated host listing counts
    'calculated_host_listings_count', 'calculated_host_listings_count_entire_homes',
    'calculated_host_listings_count_private_rooms', 'calculated_host_listings_count_shared_rooms',
    # Legal
    'license'
]

existing = [c for c in pass1_cols if c in df.columns]
df.drop(columns=existing, inplace=True)
print(f'Pass 1: Dropped {len(existing)} columns → {df.shape[1]} columns remaining')

## Pass 2 — Drop Columns Irrelevant to Business Analysis
For the **business perspective** (opening a new hotel), we remove:
- Personal host profile fields (`host_name`, `host_about`, `host_has_profile_pic`, etc.)
- Free-text descriptions that can't be used in quantitative analysis
- Redundant text/date fields already covered by numeric equivalents

In [ ]:
pass2_cols = [
    # Personal host profile — not useful for business location/pricing decisions
    'host_response_rate',
    'host_total_listings_count',
    'host_has_profile_pic',
    'host_verifications',
    'host_name',
    'host_about',             # 45% missing, free-text bio
    'host_location',          # 23% missing, redundant with listing lat/lon
    'host_identity_verified',
    # Free-text fields — cannot be used in statistical/quantitative analysis
    'description',
    'name',                   # listing title
    'neighborhood_overview',  # 48.5% missing, free text
    # Redundant date / availability fields
    'first_review',           # date of first review, not strategic
    'last_review',            # date of last review, not strategic
    'has_availability',       # boolean flag, redundant with availability_30/60/90/365
    'bathrooms_text',         # text version of bathrooms, redundant with numeric 'bathrooms'
    'number_of_reviews_l30d', # too short-term for business planning
    'availability_eoy',      # end-of-year calendar artifact
]

existing = [c for c in pass2_cols if c in df.columns]
df.drop(columns=existing, inplace=True)
print(f'Pass 2: Dropped {len(existing)} columns → {df.shape[1]} columns remaining')

## Final Dataset Summary
The remaining **38 columns** are all directly relevant to answering business questions:
- **Location**: `neighbourhood_cleansed`, `latitude`, `longitude`
- **Pricing & Revenue**: `price`, `estimated_revenue_l365d`
- **Property Details**: `property_type`, `room_type`, `bedrooms`, `beds`, `bathrooms`, `accommodates`, `amenities`
- **Demand Signals**: `estimated_occupancy_l365d`, `availability_30/60/90/365`, `reviews_per_month`, `number_of_reviews`
- **Quality Scores**: `review_scores_rating`, `review_scores_cleanliness`, `review_scores_location`, etc.
- **Host Competitiveness**: `host_is_superhost`, `host_response_rate`, `host_acceptance_rate`, `instant_bookable`

In [ ]:
# Display all remaining columns
print(f'Final dataset: {df.shape[0]} rows × {df.shape[1]} columns\n')
print('Remaining columns:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2d}. {col}')

In [ ]:
# Save the cleaned dataset
save_path = '../data/raw/airbnb_listing.csv'
df.to_csv(save_path, index=False)
print(f'\nCleaned dataset saved to {save_path}')